In [ ]:
import sys
sys.path.append("../")

In [ ]:
# imports
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import  Masking, SimpleRNN, GRU, LSTM, Bidirectional, Dense
import numpy as np


2025-04-23 08:19:15.118263: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-04-23 08:19:15.118303: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-04-23 08:19:15.118929: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-23 08:19:15.123376: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Data Wrangling

### 1. load full DF

In [ ]:
response = 'TYPE'
X_numerical = ['INCL', 'RAAN', 'ECC', 'ARG_PER', 'MEAN_MOTION', 'SMA_KM','APOGEE_KM', 'PERIGEE_KM', 'MEAN_MOTION_1ST_DER']

df_final = pd.read_csv('../data/large/final_df_v1.csv') 
df_final = df_final.sort_values(by=['NUMBER', 'EPOCH'])
df_final.head(2)

,NUMBER,NAME,TYPE,RCS,IS_CURRENT,REGIME,EPOCH,INCL,RAAN,ECC,...,MEAN_MOTION,SMA_KM,APOGEE_KM,PERIGEE_KM,MEAN_MOTION_1ST_DER,MEAN_MOTION_2ND_DER,B_STAR,COUNTRY,LINE1,LINE2
247980,5,VANGUARD 1,PAYLOAD,SMALL,1,LEO,23074.130708,34.2509,25.1938,0.184596,...,10.850716,8618.457913,3831.293356,649.422471,0.000002,0.0,0.000218,US,1 00005U 58002B 23074.13070820 .00000180 0...,2 00005 34.2509 25.1938 1845963 176.1008 185...
267637,5,VANGUARD 1,PAYLOAD,SMALL,1,LEO,23091.062771,34.2627,333.0139,0.184703,...,10.850768,8618.430495,3832.183910,648.477080,0.000002,0.0,0.000188,US,1 00005U 58002B 23091.06277074 .00000170 0...,2 00005 34.2627 333.0139 1847034 252.3237 86...


### 2. Filter by Minimum number of states

In [3]:
min_num_states = 20
rso2count = df_final['NUMBER'].value_counts()
valid_rsos = rso2count[rso2count.values >= min_num_states].index
df_filtered = df_final[df_final['NUMBER'].isin(valid_rsos)]
df_filtered.reset_index(drop=True,inplace=True)
df_filtered.head(2)

,NUMBER,NAME,TYPE,RCS,IS_CURRENT,REGIME,EPOCH,INCL,RAAN,ECC,...,MEAN_MOTION,SMA_KM,APOGEE_KM,PERIGEE_KM,MEAN_MOTION_1ST_DER,MEAN_MOTION_2ND_DER,B_STAR,COUNTRY,LINE1,LINE2
0,5,VANGUARD 1,PAYLOAD,SMALL,1,LEO,23074.130708,34.2509,25.1938,0.184596,...,10.850716,8618.457913,3831.293356,649.422471,0.000002,0.0,0.000218,US,1 00005U 58002B 23074.13070820 .00000180 0...,2 00005 34.2509 25.1938 1845963 176.1008 185...
1,5,VANGUARD 1,PAYLOAD,SMALL,1,LEO,23091.062771,34.2627,333.0139,0.184703,...,10.850768,8618.430495,3832.183910,648.477080,0.000002,0.0,0.000188,US,1 00005U 58002B 23091.06277074 .00000170 0...,2 00005 34.2627 333.0139 1847034 252.3237 86...


### 3. reshape, scale, encode, and train-test-val split

In [4]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(df_filtered[response])

raw_data = []
raw_labels = []

for rso in valid_rsos:
    df_tmp = df_filtered[df_filtered['NUMBER']==rso]
    raw_data.append(df_tmp[X_numerical].values)
    raw_labels.append(label_encoder.transform(df_tmp[response])[0])

scaler = StandardScaler().fit(np.concatenate(raw_data, axis=0))
scaled_data = [scaler.transform(seq) for seq in raw_data]
padded_data = pad_sequences(scaled_data, padding='pre', dtype='float32')
labels = tf.keras.utils.to_categorical(raw_labels, num_classes=3)

X_temp, X_test, y_temp, y_test = train_test_split(
    padded_data, labels, test_size=0.2, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42
)    

## Sequential Models

In [5]:
BATCH_SIZE = 16
DEFAULT_UNITS = 128
DEFAULT_EPOCHS = 10

### 1. RNN - Basic

In [6]:

train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(100).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


model = Sequential()
model.add(Masking(mask_value=0., input_shape=(None, len(X_numerical))))
model.add(SimpleRNN(units=DEFAULT_UNITS, return_sequences=False))
model.add(Dense(3, activation='softmax'))


model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

model.fit(
    train_dataset, 
    epochs=DEFAULT_EPOCHS, 
    validation_data=val_dataset,
    batch_size=16,         
)

test_loss, test_accuracy = model.evaluate(test_dataset)
print(f"\n✅ Test Accuracy: {test_accuracy:.4f}")

2025-04-23 08:19:54.666982: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 20763 MB memory:  -> device: 0, name: NVIDIA L4, pci bus id: 0000:3e:00.0, compute capability: 8.9


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 masking (Masking)           (None, None, 9)           0         
                                                                 
 simple_rnn (SimpleRNN)      (None, 128)               17664     
                                                                 
 dense (Dense)               (None, 3)                 387       
                                                                 
Total params: 18051 (70.51 KB)
Trainable params: 18051 (70.51 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/10


2025-04-23 08:19:56.671993: I external/local_xla/xla/service/service.cc:168] XLA service 0x1466f7997c70 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-04-23 08:19:56.672029: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA L4, Compute Capability 8.9
2025-04-23 08:19:56.676838: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-04-23 08:19:56.695853: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907
I0000 00:00:1745410796.761606   45056 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


927/927 [==============================] - 31s 32ms/step - loss: 0.6227 - accuracy: 0.7376 - val_loss: 0.5143 - val_accuracy: 0.8038
Epoch 2/10
927/927 [==============================] - 29s 31ms/step - loss: 0.5630 - accuracy: 0.7695 - val_loss: 0.6243 - val_accuracy: 0.7318
Epoch 3/10
927/927 [==============================] - 29s 31ms/step - loss: 0.5269 - accuracy: 0.7881 - val_loss: 0.4356 - val_accuracy: 0.8292
Epoch 4/10
927/927 [==============================] - 29s 32ms/step - loss: 0.4993 - accuracy: 0.8021 - val_loss: 0.4542 - val_accuracy: 0.8276
Epoch 5/10
927/927 [==============================] - 29s 32ms/step - loss: 0.5424 - accuracy: 0.7778 - val_loss: 0.7518 - val_accuracy: 0.6951
Epoch 6/10
927/927 [==============================] - 29s 31ms/step - loss: 0.6416 - accuracy: 0.7311 - val_loss: 0.5540 - val_accuracy: 0.7679
Epoch 7/10
927/927 [==============================] - 29s 31ms/step - loss: 0.5664 - accuracy: 0.7614 - val_loss: 0.5262 - val_accuracy: 0.7809
Epo

### 2. RNN - Bidirectional

In [7]:
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(100).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


model = Sequential()
model.add(Masking(mask_value=0., input_shape=(None, len(X_numerical))))
model.add(Bidirectional(SimpleRNN(units=DEFAULT_UNITS, return_sequences=False)))

model.add(Dense(3, activation='softmax'))


model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

model.fit(
    train_dataset, 
    epochs=DEFAULT_EPOCHS, 
    validation_data=val_dataset,
    batch_size=16,         
)

test_loss, test_accuracy = model.evaluate(test_dataset)
print(f"\n✅ Test Accuracy: {test_accuracy:.4f}")

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 masking_1 (Masking)         (None, None, 9)           0         
                                                                 
 bidirectional (Bidirection  (None, 256)               35328     
 al)                                                             
                                                                 
 dense_1 (Dense)             (None, 3)                 771       
                                                                 
Total params: 36099 (141.01 KB)
Trainable params: 36099 (141.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/10
927/927 [==============================] - 55s 58ms/step - loss: 0.6246 - accuracy: 0.7322 - val_loss: 0.5703 - val_accuracy: 0.7493
Epoch 2/10
927/927 [==============================] - 53s 58ms/step - 

### 3. GRU - Basic

In [8]:

train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(100).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


model = Sequential()
model.add(Masking(mask_value=0., input_shape=(None, len(X_numerical))))
model.add(GRU(units=DEFAULT_UNITS, return_sequences=False))
model.add(Dense(3, activation='softmax'))


model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

model.fit(
    train_dataset, 
    epochs=DEFAULT_EPOCHS, 
    validation_data=val_dataset,
    batch_size=16,         
)

test_loss, test_accuracy = model.evaluate(test_dataset)
print(f"\n✅ Test Accuracy: {test_accuracy:.4f}")

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 masking_2 (Masking)         (None, None, 9)           0         
                                                                 
 gru (GRU)                   (None, 128)               53376     
                                                                 
 dense_2 (Dense)             (None, 3)                 387       
                                                                 
Total params: 53763 (210.01 KB)
Trainable params: 53763 (210.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/10
927/927 [==============================] - 57s 59ms/step - loss: 0.5773 - accuracy: 0.7620 - val_loss: 0.4718 - val_accuracy: 0.8106
Epoch 2/10
927/927 [==============================] - 54s 58ms/step - loss: 0.4622 - accuracy: 0.8199 - val_loss: 0.4020 - val_accuracy:

### 4. GRU - Bidirectional 

In [9]:
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(100).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


model = Sequential()
model.add(Masking(mask_value=0., input_shape=(None, len(X_numerical))))
model.add(Bidirectional(GRU(units=DEFAULT_UNITS, return_sequences=False)))

model.add(Dense(3, activation='softmax'))


model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

model.fit(
    train_dataset, 
    epochs=DEFAULT_EPOCHS, 
    validation_data=val_dataset,
    batch_size=16,         
)

test_loss, test_accuracy = model.evaluate(test_dataset)
print(f"\n✅ Test Accuracy: {test_accuracy:.4f}")

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 masking_3 (Masking)         (None, None, 9)           0         
                                                                 
 bidirectional_1 (Bidirecti  (None, 256)               106752    
 onal)                                                           
                                                                 
 dense_3 (Dense)             (None, 3)                 771       
                                                                 
Total params: 107523 (420.01 KB)
Trainable params: 107523 (420.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/10


2025-04-23 08:42:55.210279: W tensorflow/core/common_runtime/type_inference.cc:339] Type inference failed. This indicates an invalid graph that escaped type checking. Error message: INVALID_ARGUMENT: expected compatible input types, but input 1:
type_id: TFT_OPTIONAL
args {
  type_id: TFT_PRODUCT
  args {
    type_id: TFT_TENSOR
    args {
      type_id: TFT_INT32
    }
  }
}
 is neither a subtype nor a supertype of the combined inputs preceding it:
type_id: TFT_OPTIONAL
args {
  type_id: TFT_PRODUCT
  args {
    type_id: TFT_TENSOR
    args {
      type_id: TFT_FLOAT
    }
  }
}

	for Tuple type infernce function 0
	while inferring type of node 'cond_35/output/_22'


927/927 [==============================] - 84s 85ms/step - loss: 0.5798 - accuracy: 0.7555 - val_loss: 0.4697 - val_accuracy: 0.8189
Epoch 2/10
927/927 [==============================] - 78s 84ms/step - loss: 0.4643 - accuracy: 0.8195 - val_loss: 0.4216 - val_accuracy: 0.8335
Epoch 3/10
927/927 [==============================] - 78s 84ms/step - loss: 0.4188 - accuracy: 0.8379 - val_loss: 0.3734 - val_accuracy: 0.8475
Epoch 4/10
927/927 [==============================] - 78s 84ms/step - loss: 0.3911 - accuracy: 0.8466 - val_loss: 0.3516 - val_accuracy: 0.8597
Epoch 5/10
927/927 [==============================] - 78s 84ms/step - loss: 0.3625 - accuracy: 0.8558 - val_loss: 0.3510 - val_accuracy: 0.8564
Epoch 6/10
927/927 [==============================] - 78s 84ms/step - loss: 0.3509 - accuracy: 0.8589 - val_loss: 0.3391 - val_accuracy: 0.8583
Epoch 7/10
927/927 [==============================] - 78s 84ms/step - loss: 0.3302 - accuracy: 0.8692 - val_loss: 0.3107 - val_accuracy: 0.8759
Epo

### 5. LSTM - Basic

In [10]:
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(100).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


model = Sequential()
model.add(Masking(mask_value=0., input_shape=(None, len(X_numerical))))
#model.add(LSTM(units=DEFAULT_UNITS, return_sequences=True))
model.add(LSTM(units=DEFAULT_UNITS, return_sequences=False))

model.add(Dense(3, activation='softmax'))

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

model.fit(
    train_dataset, 
    epochs=DEFAULT_EPOCHS, 
    validation_data=val_dataset,
    batch_size=16,         
)

test_loss, test_accuracy = model.evaluate(test_dataset)
print(f"\n✅ Test Accuracy: {test_accuracy:.4f}")

Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 masking_4 (Masking)         (None, None, 9)           0         
                                                                 
 lstm (LSTM)                 (None, 128)               70656     
                                                                 
 dense_4 (Dense)             (None, 3)                 387       
                                                                 
Total params: 71043 (277.51 KB)
Trainable params: 71043 (277.51 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/10
927/927 [==============================] - 56s 58ms/step - loss: 0.5462 - accuracy: 0.7790 - val_loss: 0.5320 - val_accuracy: 0.7833
Epoch 2/10
927/927 [==============================] - 52s 56ms/step - loss: 0.4352 - accuracy: 0.8295 - val_loss: 0.4238 - val_accuracy:

### 6. LSTM - Bidirectional

In [11]:
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(100).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


model = Sequential()
model.add(Masking(mask_value=0., input_shape=(None, len(X_numerical))))
model.add(Bidirectional(LSTM(units=DEFAULT_UNITS, return_sequences=False)))

model.add(Dense(3, activation='softmax'))

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

model.fit(
    train_dataset, 
    epochs=DEFAULT_EPOCHS, 
    validation_data=val_dataset,
    batch_size=16,         
)

test_loss, test_accuracy = model.evaluate(test_dataset)
print(f"\n✅ Test Accuracy: {test_accuracy:.4f}")

Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 masking_5 (Masking)         (None, None, 9)           0         
                                                                 
 bidirectional_2 (Bidirecti  (None, 256)               141312    
 onal)                                                           
                                                                 
 dense_5 (Dense)             (None, 3)                 771       
                                                                 
Total params: 142083 (555.01 KB)
Trainable params: 142083 (555.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/10
927/927 [==============================] - 82s 84ms/step - loss: 0.5282 - accuracy: 0.7837 - val_loss: 0.4301 - val_accuracy: 0.8273
Epoch 2/10
927/927 [==============================] - 75s 81ms/step 